# Next-Token Prediction Explorer

**Requires GPU runtime:** Runtime → Change runtime type → GPU

Run every cell from top to bottom. The last cell launches a public API.

In [ ]:
!pip install transformers accelerate torch gradio --quiet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")

model.eval()
print("Model loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.


In [ ]:
def predict_next_tokens(text, temperature=1.0, top_k=50, top_n=10, system_prompt=""):
    """Given an incomplete sentence, return the top N most likely next tokens."""

    if system_prompt.strip():
        full_text = system_prompt.strip() + "\n\n" + text
    else:
        full_text = text

    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    next_token_logits = outputs.logits[0, -1, :]

    # Apply temperature
    if temperature > 0:
        next_token_logits = next_token_logits / temperature

    # Apply top-k filtering
    if top_k > 0:
        top_k = min(top_k, next_token_logits.size(-1))
        indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
        next_token_logits[indices_to_remove] = -float('Inf')

    probs = torch.softmax(next_token_logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, top_n)

    results = []
    for prob, token_id in zip(top_probs, top_ids):
        tid = token_id.item()
        tok = tokenizer.convert_ids_to_tokens(tid)

        # Llama marks a leading space with "▁"; byte tokens like <0x0A> (newline) need real decoding
        if tok.startswith("<0x"):
            piece = tokenizer.decode(tid)
        else:
            piece = tok.replace("▁", " ")

        is_end = (tid == tokenizer.eos_token_id) or ("\n" in piece)

        results.append({
            "word": "<end>" if tid == tokenizer.eos_token_id else piece,
            "id": tid,
            "probability": round(prob.item(), 4),
            "is_end": is_end,
        })

    return results

print("predict_next_tokens() ready.")

predict_next_tokens() ready.


In [ ]:
# Quick test
sentence = "Data visualization empowers users to"
predictions = predict_next_tokens(sentence)

print(f'Incomplete sentence: "{sentence}"')
print(f'\nTop {len(predictions)} most likely next words:\n')
for i, p in enumerate(predictions, 1):
    bar = '\u2588' * int(p['probability'] * 100)
    print(f"  {i:>2}. {p['word']:<15} {p['probability']*100:>6.2f}%  {bar}")

Incomplete sentence: "Data visualization empowers users to"

Top 10 most likely next words:

   1.  easily          10.85%  ██████████
   2.  quickly         10.68%  ██████████
   3.  explore          9.95%  █████████
   4.  create           6.28%  ██████
   5.  make             5.54%  █████
   6.  see              5.33%  █████
   7.  analyze          5.12%  █████
   8.  understand       4.85%  ████
   9.  visual           4.49%  ████
  10.  better           4.02%  ████


In [ ]:
import gradio as gr

def api_predict(text, temperature, top_k, system_prompt):
    results = predict_next_tokens(
        text,
        temperature=temperature,
        top_k=int(top_k),
        top_n=10,
        system_prompt=system_prompt
    )
    return {"model": model_name, "predictions": results}

demo = gr.Interface(
    fn=api_predict,
    inputs=[
        gr.Textbox(label="Incomplete sentence"),
        gr.Slider(0.01, 2.0, value=1.0, step=0.01, label="Temperature"),
        gr.Slider(1, 100, value=50, step=1, label="Top-K"),
        gr.Textbox(label="System prompt", value=""),
    ],
    outputs=gr.JSON(label="Predictions"),
    title="Next-Token Prediction API",
    api_name="predict",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b60910bc0b8d0e5b1d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
